# Build 4 — DAGSNet alone, no extractor and no CMSO

This is the **ablation the first three builds could not answer**: how much of the score comes
from §4.8 (DWT → ViT → GAT → Eq. 28 fusion) and §4.9 (CMSO channel selection), and how much
from DAGSNet by itself?

| | build 2/3 | **build 4** |
|---|---|---|
| §4.8 extractor, Eq. 19–27 | DWT → ViT×2 → GAT | **removed** |
| Eq. 28 fusion | 6 + 128 + 128 = 262 channels | **removed** |
| §4.9 CMSO | 133 of 262 channels | **removed** |
| DAGSNet input | `(B, 133, 11)` | `(B, 6, 11)` — the 66 raw features |
| §4.10 DAGSNet, Eq. 38–48 | DenseNet + GoogleNet + AlexNet + SqueezeNet | **identical** |
| trainable parameters | 727,952 | **395,024** |

**Input layout.** DAGSNet consumes `(B, C, L)`. The 66 standardised features are reshaped to
11 positions × 6 channels — *the same grid `PatchEmbed` chunks the DWT output into*, fed raw.
Only the stem's input width changes (133 → 6); every branch keeps its exact geometry, so a
difference in results is the extractor's contribution and not a different CNN.

Two layouts were measured and rejected. `(B, 1, 66)` — one channel, length 66 — is **1.3×
slower than the full pipeline** (66 convolution positions instead of 11) and needs 10.29 h.
`(B, 66, 1)` is fastest but convolutions over a length-1 axis degenerate to 1×1, which quietly
turns DAGSNet into an MLP.

**Every hyperparameter is unchanged from builds 1–3**: `stem_ch=96`, `dense_growth=32`,
`dense_layers=3`, `incep_modules=2`, `fire_modules=3`, `dropout=0.1`, `patch_len=6`,
batch 4,096, 50 rounds × 1 epoch, Adam `lr=1e-3` with 1 warmup round then cosine,
`weight_decay=1e-4`, cross-entropy. The parameter count falls to 395,024 because ViT and GAT
are genuinely gone, not because anything was resized.

**Budget.** Measured locally: 1.89× faster than the full pipeline per training step. Against
c50's measured 565.7 s/round that projects to **299.3 s/round → 4.16 h for 50 rounds**, plus
~3.2 min of parquet prep (build 2's other 77 minutes were CMSO, which does not exist here).

**Nothing is inherited.** `channel_mask.json` and `extractor_init.pt` describe a projection
this build does not contain. They are deliberately *not* loaded.

### The three NaN mechanisms this build guards against

Every one of them cost a real run.

1. **fp16 cross-entropy.** A softmax over 16 half-precision logits can round a class
   probability to exactly 0, and `log(0)` is `-inf`. The loss is therefore computed in fp32
   **outside** `autocast`.
2. **A tripwire that fires too late.** Build 2 run 1 went non-finite at round 25 and trained 20
   more rounds — 4.1 h of GPU, 20 NaN checkpoints, and `last.pt` overwritten with NaN weights
   so the run could not even be resumed. The check here runs on the `all_reduce`d loss
   **before** evaluation and **before** the checkpoint write.
3. **Silent skipping.** `GradScaler` discards every non-finite step and reports nothing, so a
   run can throw away most of its batches while printing a falling loss. Skipped steps are
   counted, logged, and abort the round if they dominate.

W&B receives a heartbeat every 250 steps and all 10 metrics after each round, so the run can
be watched while it is still running — Kaggle itself publishes nothing until the kernel stops.

In [ ]:
# ── Cell 2 · environment probe ───────────────────────────────────────────────
import os, sys, math, json, time, glob, shutil, platform
import numpy as np, torch

NB_T0 = time.time()          # session wall-clock origin; the round budget counts from here

print("python", platform.python_version(), "| torch", torch.__version__, "| cuda", torch.version.cuda)
n = torch.cuda.device_count(); print("GPUs:", n)
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} {p.name} {p.total_memory/2**30:.1f} GB sm_{p.major}{p.minor}")
print("cpu count:", os.cpu_count())
print("RAM GB:", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1))
for d in ("/kaggle/working", "/kaggle/temp"):
    os.makedirs(d, exist_ok=True)
    print(f"free on {d}:", round(shutil.disk_usage(d).free / 2**30, 1), "GB")

assert n == 2, f"Expected 2x T4 — got {n}. Set Accelerator to 'GPU T4 x2' and restart."
assert torch.cuda.get_device_properties(0).major == 7, \
    "Expected Turing (sm75). fp16 AMP settings below assume no bf16 and no TF32."

# proj package: %%writefile keeps the code visible in the notebook AND importable by
# the processes mp.spawn creates — functions defined in bare cells are not.
import pathlib
PKG = pathlib.Path("/kaggle/working/proj"); PKG.mkdir(parents=True, exist_ok=True)
(PKG / "__init__.py").touch()
sys.path.insert(0, "/kaggle/working")
print("proj package ready")

In [ ]:
# ── Cell 3 · CFG — the single knob cell ──────────────────────────────────────
from dataclasses import dataclass, asdict

@dataclass
class _CFG:
    run_name: str = "edl_cmso_v4_dagsnet"

    # ---- identical to builds 1-3; the ONLY thing that differs in build 4 is that
    # ---- §4.8 and §4.9 are absent. Nothing here was retuned. --------------------
    rounds: int           = 50
    epochs_per_round: int = 1
    batch_per_gpu: int    = 2048    # global = 2048 x 2 ranks x 1 = 4096
    grad_accum: int       = 1
    lr: float             = 1e-3    # Table 1: "0.001 with adaptive decay"
    weight_decay: float   = 1e-4
    warmup_rounds: int    = 1       # then cosine to zero

    # DAGSNet geometry — every value carried over unchanged from build 2
    patch_len: int    = 6           # 66 = 11 x 6 exactly -> (B, 6, 11), no padding
    dropout: float    = 0.1
    stem_ch: int      = 96
    dense_growth: int = 32
    dense_layers: int = 3
    incep_modules: int= 2
    fire_modules: int = 3

    # runtime
    # Quota, not the 12 h session cap, is binding: 4.64 h of GPU remain before the
    # 2026-09-05 refresh. Training is projected at 4.16 h and prep at ~0.06 h, so the
    # clean stop sits at 4.35 h — early enough that the closing cells run and the output
    # commits with quota still in hand. A run that overruns stops cleanly and resumes.
    max_hours: float  = 2.00
    nccl_timeout_min: int = 60
    world_size: int   = 2
    amp: bool         = True        # fp16 + GradScaler; T4 is Turing, no bf16
    clip: float       = 1.0
    compile_model: bool = True      # measured 1.34x on these T4s in build 2 run 3
    eval_batch: int   = 16384
    seed: int         = 42
    resume: bool      = True
    num_classes: int  = -1          # from meta.json
    n_features: int   = -1          # from meta.json — all 66, nothing is selected away
    save_prob_every: int = 0        # best + final round only

    # observability — Kaggle publishes nothing until the kernel stops
    heartbeat_every: int   = 250    # optimizer steps
    abort_after: int       = 1500   # grace period before the skip-rate abort can fire
    abort_skip_pct: float  = 60.0
    wandb_entity: str  = "21522798-uit"
    wandb_project: str = "edl-cmso"
    wandb_run_id: str  = "edl-cmso-v4-dagsnet-r46"   # a SEPARATE run: the first r46
    # attempt resumed the original run id and wrote a second round-0 curve into it

CFG = _CFG()
GLOBAL_BATCH = CFG.batch_per_gpu * CFG.world_size * CFG.grad_accum
print(f"global batch = {CFG.batch_per_gpu} x {CFG.world_size} x {CFG.grad_accum} = {GLOBAL_BATCH}")
print(f"rounds = {CFG.rounds} x {CFG.epochs_per_round} epoch -> "
      f"{CFG.rounds * CFG.epochs_per_round} passes over train")
print(f"DAGSNet input = (B, {CFG.patch_len}, 66/{CFG.patch_len}) — raw features, no extractor")
print(f"session budget = {CFG.max_hours} h (quota-limited)")
print(f"precision      = {'fp16 AMP + GradScaler' if CFG.amp else 'fp32'}   "
      f"torch.compile = {CFG.compile_model}")

# This notebook exists only to finish rounds 46-49 of the completed build 4 run.
# Without an inherited checkpoint it has no reason to run at all.
REQUIRE_RESUME = True


In [ ]:
# ── Cell 4 · paths, W&B (fail fast), run scaffold, resume probe ──────────────
from pathlib import Path

CACHE = Path("/kaggle/temp/veremi_cache"); CACHE.mkdir(parents=True, exist_ok=True)
RUNS  = Path("/kaggle/working/runs");      RUNS.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS / CFG.run_name
for s in ("checkpoints", "metrics", "preds", "confusion", "reports", "logs"):
    (RUN_DIR / s).mkdir(parents=True, exist_ok=True)

# ---- W&B authenticates HERE, before the parquet pass ------------------------
# A missing credential or no Internet must cost seconds, not the minutes of data
# preparation that precede round 0. Only the credential is checked in this process; the
# run itself is opened by rank 0 inside the worker, so exactly one process owns it.
try:
    import wandb
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb>=0.29,<1"])
    import wandb
# BEGIN INLINE WANDB CREDENTIAL — owner-authorized private notebook
wandb.login(key="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", relogin=True, verify=True)
# END INLINE WANDB CREDENTIAL
print("W&B authenticated; run will be opened by rank 0 as "
      f"{CFG.wandb_entity}/{CFG.wandb_project}/{CFG.wandb_run_id}")
(RUN_DIR / "wandb_run.json").write_text(json.dumps({
    "entity": CFG.wandb_entity, "project": CFG.wandb_project, "run_id": CFG.wandb_run_id,
    "run_path": f"{CFG.wandb_entity}/{CFG.wandb_project}/{CFG.wandb_run_id}",
    "url": f"https://wandb.ai/{CFG.wandb_entity}/{CFG.wandb_project}/runs/{CFG.wandb_run_id}",
}, indent=2))   # identity only — never the secret

print("-- /kaggle/input --")
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        depth = root.count("/") - 2
        if depth > 3: dirs[:] = []; continue
        print("  " * depth + Path(root).name + "/" +
              (f"   [{len(files)} files]" if files else ""))
else:
    print("  /kaggle/input does not exist — no data source is attached at all")

cands = sorted(p for p in glob.glob("/kaggle/input/**/train", recursive=True)
               if os.path.isdir(p) and glob.glob(os.path.join(p, "*.parquet")))
assert cands, ("Dataset not attached, or attached without parquet under a 'train/' dir. "
               "Add 'odixe0502/veremi-nextgen2026-centralized' via + Add Input, then re-run.")
DATA = Path(cands[0]).parent
print("\ndataset root:", DATA)
for f in sorted(os.listdir(DATA)):
    p = DATA / f
    extra = f" ({len(os.listdir(p))} files)" if p.is_dir() else f" ({p.stat().st_size/2**20:.1f} MiB)"
    print("  ", f, extra)

# Kaggle nests an attached notebook output at /kaggle/input/notebooks/<owner>/<slug>/,
# which is THREE levels deep. The first attempt at this resume globbed at most two levels
# ("/kaggle/input/*/*/runs/..."), found nothing, and — because a missing checkpoint was
# only a warning — silently retrained from round 0 for the whole 1.9 h budget. Search
# recursively, the same way the dataset above is found, and make absence fatal.
RESUME_GLOB = f"/kaggle/input/**/runs/{CFG.run_name}/checkpoints/last.pt"

def probe():
    last = RUN_DIR / "checkpoints" / "last.pt"
    imported = sorted(glob.glob(RESUME_GLOB, recursive=True)) \
             + sorted(glob.glob("/kaggle/input/**/checkpoints/last.pt", recursive=True))
    hist = RUN_DIR / "metrics" / "history.csv"
    print("\n-- resume probe --")
    print("  local last.pt      :", "yes" if last.exists() else "no")
    print("  attached last.pt   :", imported[0] if imported else "none")
    print("  local history.csv  :", f"{sum(1 for _ in open(hist))-1} rows" if hist.exists() else "absent")
    print("  CMSO / extractor   : not applicable — build 4 has neither")
    if REQUIRE_RESUME and not (last.exists() or imported):
        print("\n  everything under /kaggle/input/notebooks:")
        for root, _, files in os.walk("/kaggle/input/notebooks"):
            for f in files:
                print("   ", os.path.join(root, f))
        raise SystemExit(
            "REQUIRE_RESUME is set and no checkpoint was found. This run exists only to "
            "continue round 45, so starting from round 0 would burn the budget for "
            "nothing. Fix the attachment or the glob above, then re-push.")
probe()

## Dataset — VeReMi NextGen, 16-class centralized

Derived from Zenodo record `19665762` (DOI `10.5281/zenodo.19665762`). One row is one
received CAM message, seen from the receiving vehicle's side.

| split | files | rows | size | feature state |
|---|---:|---:|---:|---|
| `train/` | 15 | 43,045,415 | 7.3 GiB | imputed **and standardised** — use as is |
| `test/` | 4 | 10,761,343 | 1.7 GiB | imputed, **not standardised** |

**A column is a model feature if and only if its name starts with `f_`** — 66 of them,
float32, grouped as `raw` 15, `position` 4, `geometry` 20, `session` 16, `rate` 5,
`profile` 6. Everything else is leakage and is dropped: 9 identifiers (`receiver_id`,
`sender_id`, `sender_alias`, `message_id`, `rcv_time_ns`, `send_time_ns`, `orig_split`,
`source_run`, `t_rel_s`), 9 raw-context columns, plus `attack_type` and `scenario`, which
are partition columns materialised as real columns and would hand the model the answer.

**The two preprocessing steps and why the notebook skips one and applies the other.**
Imputation came *first*, before the split: every non-finite cell was filled with a constant
`0.0` while the features were one continuous stream. A constant fill needs no statistic, so
its position in the order cannot leak test into train. The 13 `zero_imputed_features` are
session deltas with no value on a session's first message — `f_first_in_session` is `1` on
exactly those rows, so the imputation stays visible to the model rather than hidden.
Standardisation came *second* and **on train only**: `mean` and `std_used` in `scaler.json`
were fitted on the 43,045,415 train rows with population variance (`ddof=0`). So `train/`
is consumed untouched, and `test/` gets that frozen transform applied at load. The cell
below asserts we never do it the other way round.

The split is by **simulation time**, with a separate cut point per (class × scenario) —
64 cut points, `rcv_time_ns < t*` to train. Nothing is split randomly, so no message leaks
from a session's future into its past. The test split is given; we do not create one.

In [ ]:
# ── Cell 6 · ONE pass over parquet: audit and materialise together ───────────
import pyarrow.parquet as pq
import pandas as pd

TRAIN_DIR, TEST_DIR = DATA / "train", DATA / "test"
train_files = sorted(glob.glob(str(TRAIN_DIR / "*.parquet")))
test_files  = sorted(glob.glob(str(TEST_DIR  / "*.parquet")))
print(f"train shards: {len(train_files)}   test shards: {len(test_files)}")

schema = pq.ParquetFile(train_files[0]).schema_arrow
all_cols     = list(schema.names)
FEATURES     = [c for c in all_cols if c.startswith("f_")]     # the ONLY rule that matters
NON_FEATURES = [c for c in all_cols if not c.startswith("f_")]
print(f"\ncolumns: {len(all_cols)}   features (f_*): {len(FEATURES)}   dropped: {len(NON_FEATURES)}")
print("dropped as identifiers / labels / raw context / partition keys:")
print("  ", NON_FEATURES)

# sidecar json shipped with the dataset
def load_json(name):
    p = DATA / name
    return json.load(open(p)) if p.exists() else None
fschema, scaler_j, labmap, manifest = (load_json(f) for f in
    ("feature_schema.json", "scaler.json", "label_mapping.json", "manifest.json"))

FEATURE_GROUPS = {}
if fschema:
    declared = fschema["feature_columns"]
    assert declared == FEATURES, "feature_schema.json disagrees with the f_* rule — stop and inspect."
    print("\nfeature_schema.json agrees with the f_* rule.")
    FEATURE_GROUPS = fschema.get("groups") or fschema.get("feature_groups") or {}
    if FEATURE_GROUPS:
        print("groups:", {k: len(v) for k, v in FEATURE_GROUPS.items()})
    if "zero_imputed_features" in fschema:
        print(f"zero-imputed session features: {len(fschema['zero_imputed_features'])}")

# Frozen class axis — the index order must be identical across rounds AND sessions.
# This dataset's label_mapping.json carries three mutually redundant views:
#   label_to_class {"0": "benign", ...}   classes [ordered]   class_to_label {name: idx}
# Read the authoritative one and assert the other two agree; a disagreement means the
# class axis is not actually frozen, which silently corrupts every per-class number.
if labmap and "label_to_class" in labmap:
    inv = {int(k): v for k, v in labmap["label_to_class"].items()}
    CLASS_NAMES = [inv[i] for i in range(len(inv))]
    if "classes" in labmap:
        assert list(labmap["classes"]) == CLASS_NAMES, \
            "label_mapping.json: classes[] disagrees with label_to_class"
    if "class_to_label" in labmap:
        assert labmap["class_to_label"] == {n: i for i, n in enumerate(CLASS_NAMES)}, \
            "label_mapping.json: class_to_label is not the inverse of label_to_class"
    print("label_mapping.json: all three views agree on the class axis.")
elif labmap and all(str(k).isdigit() for k in labmap):
    inv = {int(k): v for k, v in labmap.items()}
    CLASS_NAMES = [inv[i] for i in range(len(inv))]
else:
    CLASS_NAMES = [f"class_{i}" for i in range(16)]
NUM_CLASSES = len(CLASS_NAMES)
print(f"\n{NUM_CLASSES} classes:"); [print(f"  {i:2d}  {n}") for i, n in enumerate(CLASS_NAMES)]

# scaler is fitted on TRAIN ONLY, ddof=0, and is applied to TEST ONLY
assert scaler_j is not None, "scaler.json missing — test cannot be standardised correctly."
SC = scaler_j["features"]
SC_MEAN = np.array([SC[c]["mean"]     for c in FEATURES], dtype=np.float32)
SC_STD  = np.array([SC[c]["std_used"] for c in FEATURES], dtype=np.float32)
assert (SC_STD > 0).all(), "a zero std_used would divide by zero"
print("\nscaler.json loaded: fitted on train, applied to test only (deviation 5).")

# ── the single pass ──────────────────────────────────────────────────────────
# Row counts come from the parquet FOOTERS — metadata only, nothing decoded — so the
# memmaps can be allocated before the one and only decode pass. The earlier version
# streamed every row group twice (once to audit, once to materialise); /kaggle/temp is
# wiped on every new session, so that second decode was paid again every session.
XTR = CACHE / "train_X.f16.npy"; YTR = CACHE / "train_y.i8.npy"
XTE = CACHE / "test_X.f16.npy";  YTE = CACHE / "test_y.i8.npy"
META  = CACHE / "meta.json"
AUDIT = CACHE / "audit.json"

def footer_rows(files):
    return sum(pq.ParquetFile(f).metadata.num_rows for f in files)

t_meta = time.time()
N_TRAIN, N_TEST = footer_rows(train_files), footer_rows(test_files)
print(f"\nrow counts from parquet footers in {time.time()-t_meta:.1f}s: "
      f"train {N_TRAIN:,}   test {N_TEST:,}")

def report(name, a):
    counts = np.asarray(a["counts"], dtype=np.int64); rows = a["rows"]
    print(f"\n{'='*72}\n{name}: {rows:,} rows x {len(FEATURES)} features\n{'='*72}")
    df = pd.DataFrame({"class": CLASS_NAMES, "count": counts,
                       "pct": (100 * counts / rows).round(4)})
    print(df.to_string(index=False))
    nz = counts[counts > 0]
    print(f"imbalance ratio: {counts.max() / max(nz.min(), 1):,.1f} : 1")
    print(f"empty classes  : {[CLASS_NAMES[i] for i in np.where(counts == 0)[0]] or 'none'}")
    print(f"NaN cells: {a['nan']}   Inf cells: {a['inf']}")
    assert a["nan"] == 0 and a["inf"] == 0, "dataset promises no NaN/Inf — it has some. Stop."
    print(f"feature min {a['min']:.3f}  max {a['max']:.3f}  mean(|mean|) {a['absmean']:.4f}")

def scan(files, n_rows, xp, yp, name, standardise):
    """Decode each row group ONCE: accumulate the audit and write the fp16 memmap
    from the same decoded block."""
    cache = json.loads(AUDIT.read_text()) if AUDIT.exists() else {}
    if xp.exists() and yp.exists() and xp.stem in cache:
        a = cache[xp.stem]
        print(f"\n{name}: cached from this session — parquet not re-read")
        report(name, a)
        return a["rows"], np.asarray(a["counts"], dtype=np.int64)

    X = np.lib.format.open_memmap(xp, mode="w+", dtype=np.float16, shape=(n_rows, len(FEATURES)))
    Y = np.lib.format.open_memmap(yp, mode="w+", dtype=np.int8,   shape=(n_rows,))
    counts = np.zeros(NUM_CLASSES, np.int64)
    nan_hits = inf_hits = 0
    mn = np.full(len(FEATURES),  np.inf, np.float64)
    mx = np.full(len(FEATURES), -np.inf, np.float64)
    ssum = np.zeros(len(FEATURES), np.float64)
    i = 0; t0 = time.time()
    for fp in files:
        for b in pq.ParquetFile(fp).iter_batches(batch_size=1_000_000,
                                                 columns=FEATURES + ["label"]):
            m = len(b["label"])
            A = np.column_stack([np.asarray(b[c], dtype=np.float32) for c in FEATURES])
            if standardise:                      # test only — frozen train statistics
                A = (A - SC_MEAN) / SC_STD
            yb = np.asarray(b["label"], dtype=np.int64)
            # --- audit, on the float32 block, before the fp16 cast ---
            counts += np.bincount(yb, minlength=NUM_CLASSES)
            nan_hits += int(np.isnan(A).sum()); inf_hits += int(np.isinf(A).sum())
            mn = np.minimum(mn, A.min(0)); mx = np.maximum(mx, A.max(0))
            ssum += A.sum(0, dtype=np.float64)
            # --- materialise, same block ---
            X[i:i+m] = A.astype(np.float16); Y[i:i+m] = yb.astype(np.int8)
            i += m
        print(f"    {Path(fp).name}: {i:,}/{n_rows:,}  ({time.time()-t0:.0f}s)", flush=True)
    assert i == n_rows, f"row count mismatch {i} != {n_rows}"
    X.flush(); Y.flush()
    a = {"rows": int(i), "counts": counts.tolist(), "nan": nan_hits, "inf": inf_hits,
         "min": float(mn.min()), "max": float(mx.max()),
         "absmean": float(np.abs(ssum / i).mean()), "seconds": round(time.time() - t0, 1)}
    cache[xp.stem] = a; AUDIT.write_text(json.dumps(cache, indent=2))
    report(name, a)
    print(f"one pass: decode + audit + write in {a['seconds']}s")
    return i, counts

N_TRAIN, CNT_TRAIN = scan(train_files, N_TRAIN, XTR, YTR,
                          "TRAIN (already standardised — untouched)", standardise=False)
N_TEST,  CNT_TEST  = scan(test_files,  N_TEST,  XTE, YTE,
                          "TEST  (scaler.json applied at load)",      standardise=True)

print("\n== train/test agreement ==")
te_schema = list(pq.ParquetFile(test_files[0]).schema_arrow.names)
print("cols only in train  :", sorted(set(all_cols) - set(te_schema)) or "none")
print("cols only in test   :", sorted(set(te_schema) - set(all_cols)) or "none")
print("labels only in test :", [CLASS_NAMES[i] for i in np.where((CNT_TEST > 0) & (CNT_TRAIN == 0))[0]] or "none")
print(f"train:test ratio    : {N_TRAIN/(N_TRAIN+N_TEST):.4f} : {N_TEST/(N_TRAIN+N_TEST):.4f}")


In [ ]:
# ── Cell 7 · freeze meta.json (the parquet pass already wrote the memmaps) ────
# fp16 rationale: 43,045,415 x 66 is 11.4 GB in float32 but 5.7 GB in float16, and each
# rank holds only its own shard -> ~2.8 GB resident on a 16 GB T4. The data is already
# standardised (mean 0, std 1), so fp16 has ample precision for it. This is what lets the
# DataLoader disappear from the training loop entirely (see perf notes in Cell 15).
CFG.num_classes = NUM_CLASSES
CFG.n_features  = len(FEATURES)

meta = {"class_names": CLASS_NAMES, "num_classes": NUM_CLASSES,
        "feature_cols": FEATURES, "dropped_cols": NON_FEATURES,
        "n_train": int(N_TRAIN), "n_test": int(N_TEST),
        "train_counts": CNT_TRAIN.tolist(), "test_counts": CNT_TEST.tolist(),
        "scaler": {"applied_to": "test_only", "fitted_on": "train_only", "ddof": 0},
        "paths": {k: str(v) for k, v in
                  dict(train_X=XTR, train_y=YTR, test_X=XTE, test_y=YTE).items()}}
META.write_text(json.dumps(meta, indent=2))
(RUN_DIR / "meta.json").write_text(json.dumps(meta, indent=2))   # survives into the output
print(f"frozen meta.json -> {META}")
print(f"train X {XTR.stat().st_size/2**30:.2f} GB   test X {XTE.stat().st_size/2**30:.2f} GB")
print(f"free on /kaggle/temp: {shutil.disk_usage('/kaggle/temp').free/2**30:.1f} GB")

# checkpoint budget: every round is kept
print(f"\nfree on /kaggle/working: {shutil.disk_usage('/kaggle/working').free/2**30:.1f} GB "
      f"(50 checkpoints + 50 y_pred arrays must fit)")


## Architecture — DAGSNet alone, Eq. (38)–(48)

The four backbones and the fusion head are **the same classes builds 1–3 use**, at the same
widths. What is gone is everything upstream of them.

$$\text{Dense: } x_{\ell} = H_{\ell}\big([x_0, x_1, \dots, x_{\ell-1}]\big) \tag{38-39}$$

$$\text{Inception: } y = \big[\,b_{1\times1}(x)\, \|\, b_{3\times3}(x)\, \|\, b_{5\times5}(x)\, \|\, b_{\text{pool}}(x)\,\big] \tag{40-41}$$

$$\text{AlexNet: } y = \mathrm{conv}\big(\mathrm{pool}(\mathrm{conv}(\mathrm{pool}(\mathrm{conv}(x))))\big) \tag{42-44}$$

$$\text{Fire: } y = \big[\,e_{1\times1}(s(x))\, \|\, e_{3\times3}(s(x))\,\big],\quad s = \text{squeeze}_{1\times1} \tag{45-46}$$

$$\hat{y} = \mathrm{softmax}\Big(W_2\,\mathrm{ReLU}\big(W_1\,[\,\overline{d}\,\|\,\overline{g}\,\|\,\overline{a}\,\|\,\overline{s}\,]\big)\Big) \tag{47-48}$$

where $\overline{\cdot}$ is global average pooling over the position axis. The input is the raw
feature matrix reshaped to $(B, 6, 11)$ — no wavelet, no attention, no graph, no channel mask.

In [ ]:
%%writefile /kaggle/working/proj/model.py
"""DAGSNet alone — Eq. (38)-(48) of Khan et al. 2025, with §4.8 and §4.9 removed.

Every class below is copied unchanged from build 2's model.py. The ONLY differences are
that EDLCMSO's dwt/patch/vit/gat/fuse are gone and the stem now takes patch_len channels
instead of |S| selected fused channels. Keeping the branches byte-identical is the whole
point: a difference in results then measures the extractor, not a different CNN.
"""
import torch, torch.nn as nn


def cbr(i, o, k):
    return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2, bias=False),
                         nn.BatchNorm1d(o), nn.ReLU(inplace=True))


class DenseNet1d(nn.Module):
    """Eq. (38)-(39): every layer consumes the concatenation of all preceding maps."""
    def __init__(self, cin, growth, layers):
        super().__init__()
        self.blocks = nn.ModuleList([cbr(cin + i * growth, growth, 3) for i in range(layers)])
        self.out_ch = cin + layers * growth

    def forward(self, x):
        for b in self.blocks:
            x = torch.cat([x, b(x)], dim=1)                     # Eq. (38)
        return x                                                # Eq. (39)


class Inception1d(nn.Module):
    """Eq. (40): parallel 1x1 / 3x3 / 5x5 / pool branches, concatenated."""
    def __init__(self, cin, c):
        super().__init__()
        self.b1 = cbr(cin, c, 1)
        self.b3 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 3))
        self.b5 = nn.Sequential(cbr(cin, c, 1), cbr(c, c, 5))
        self.bp = nn.Sequential(nn.MaxPool1d(3, 1, 1), cbr(cin, c, 1))
        self.out_ch = 4 * c

    def forward(self, x):
        return torch.cat([self.b1(x), self.b3(x), self.b5(x), self.bp(x)], dim=1)


class GoogleNet1d(nn.Module):
    """Eq. (40)-(41): stacked inception modules."""
    def __init__(self, cin, modules_n, c=32):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Inception1d(ch, c); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class AlexNet1d(nn.Module):
    """Eq. (42)-(44): conv+ReLU stack with max pooling. The position axis is only k long,
    so pooling is ceil_mode to avoid collapsing it to zero."""
    def __init__(self, cin, ch=128):
        super().__init__()
        self.net = nn.Sequential(
            cbr(cin, ch, 3), nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3),  nn.MaxPool1d(2, ceil_mode=True),
            cbr(ch, ch, 3))
        self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class Fire1d(nn.Module):
    """Eq. (45)-(46): 1x1 squeeze feeding parallel 1x1 and 3x3 expands."""
    def __init__(self, cin, sq, ex):
        super().__init__()
        self.squeeze = cbr(cin, sq, 1)                          # Eq. (46)
        self.e1 = cbr(sq, ex, 1)
        self.e3 = cbr(sq, ex, 3)
        self.out_ch = 2 * ex

    def forward(self, x):
        s = self.squeeze(x)
        return torch.cat([self.e1(s), self.e3(s)], dim=1)       # Eq. (45)


class SqueezeNet1d(nn.Module):
    def __init__(self, cin, modules_n, sq=32, ex=48):
        super().__init__()
        mods, ch = [], cin
        for _ in range(modules_n):
            m = Fire1d(ch, sq, ex); mods.append(m); ch = m.out_ch
        self.net = nn.Sequential(*mods); self.out_ch = ch

    def forward(self, x):
        return self.net(x)


class DAGSNet(nn.Module):
    """Eq. (38)-(48) applied directly to the raw feature matrix.

    The 66 standardised columns become (B, patch_len, k) with k = 66 / patch_len = 11:
    the same 11-position grid PatchEmbed chunks the DWT output into, fed raw. Two other
    layouts were measured and rejected — (B,1,66) is 1.3x SLOWER than the full pipeline
    because every convolution then runs over 66 positions instead of 11, and (B,66,1)
    degenerates every convolution to 1x1, turning DAGSNet into an MLP."""

    def __init__(self, cfg, n_features):
        super().__init__()
        self.patch_len = cfg["patch_len"]
        assert n_features % self.patch_len == 0, \
            f"{n_features} features do not divide into patches of {self.patch_len}"
        self.k = n_features // self.patch_len                   # 11
        cin = self.patch_len                                    # 6 channels

        s = cfg["stem_ch"]
        self.stems = nn.ModuleList([cbr(cin, s, 1) for _ in range(4)])
        self.dense   = DenseNet1d(s, cfg["dense_growth"], cfg["dense_layers"])
        self.google  = GoogleNet1d(s, cfg["incep_modules"])
        self.alex    = AlexNet1d(s)
        self.squeeze = SqueezeNet1d(s, cfg["fire_modules"])
        comb = self.dense.out_ch + self.google.out_ch + self.alex.out_ch + self.squeeze.out_ch

        self.head = nn.Sequential(                              # Eq. (48)
            nn.LayerNorm(comb), nn.Dropout(cfg["dropout"]),
            nn.Linear(comb, 256), nn.ReLU(inplace=True),
            nn.Dropout(cfg["dropout"]), nn.Linear(256, cfg["num_classes"]))

    def forward(self, x):                                       # (B, n_features)
        Fm = x.view(x.shape[0], self.k, self.patch_len).transpose(1, 2)   # (B, 6, 11)
        feats = [gp(stem(Fm)) for stem, gp in
                 zip(self.stems, [self.dense, self.google, self.alex, self.squeeze])]
        pooled = [f.mean(dim=-1) for f in feats]                # global average pool
        return self.head(torch.cat(pooled, dim=1))              # Eq. (47) -> (48)


def build_model(cfg, n_features):
    return DAGSNet(cfg, n_features)

## Loss — categorical cross-entropy (Table 1)

With $C=16$ classes, $y_i$ the true class of sample $i$ and $p_{i,c}$ the softmax
probability the model assigns to class $c$:

$$\mathcal{L} = -\frac{1}{B}\sum_{i=1}^{B}\sum_{c=1}^{C}\mathbb{1}[y_i = c]\,\log p_{i,c}
              = -\frac{1}{B}\sum_{i=1}^{B}\log p_{i,y_i}$$

`nn.CrossEntropyLoss` applies the log-softmax internally, so the model returns raw logits
(Eq. 48 with $\sigma$ folded into the loss) — numerically the right way round under fp16 AMP.

**Unweighted, deliberately.** The paper specifies plain categorical cross-entropy and
weights nothing, and the class imbalance here is 41:1 — well inside the range where
reweighting is a modelling choice rather than a necessity. Adding class weights would be a
silent deviation that changes exactly the metric being reported (`f1_macro`), so it is not
added. `f1_macro` is where any imbalance damage will show up.

## The 10 metrics

Computed on the **entire** 10,761,343-row test set after every round, by rank 0, from a
single pass. Notation: $C=16$ classes, $N$ test samples, $n_c$ true samples of class $c$.

$$\text{Accuracy} = \frac{1}{N}\sum_{i=1}^{N}\mathbb{1}[\hat{y}_i = y_i] = \frac{\sum_c TP_c}{N}$$

Per class $P_c = \dfrac{TP_c}{TP_c + FP_c}$, $R_c = \dfrac{TP_c}{TP_c + FN_c}$, $F1_c = \dfrac{2 P_c R_c}{P_c + R_c}$.

$$P_{\text{macro}} = \frac{1}{C}\sum_c P_c \qquad P_{\text{micro}} = \frac{\sum_c TP_c}{\sum_c (TP_c + FP_c)} \qquad P_{\text{weighted}} = \sum_c \frac{n_c}{N}P_c$$

$$R_{\text{macro}} = \frac{1}{C}\sum_c R_c \qquad R_{\text{micro}} = \frac{\sum_c TP_c}{\sum_c (TP_c + FN_c)} \qquad R_{\text{weighted}} = \sum_c \frac{n_c}{N}R_c$$

$$F1_{\text{macro}} = \frac{1}{C}\sum_c F1_c \qquad F1_{\text{micro}} = \frac{2P_{\text{micro}}R_{\text{micro}}}{P_{\text{micro}} + R_{\text{micro}}} \qquad F1_{\text{weighted}} = \sum_c \frac{n_c}{N}F1_c$$

**An identity to expect, not to debug.** In single-label multi-class classification every
prediction is exactly one class, so $\sum_c FP_c = \sum_c FN_c$ and therefore

$$P_{\text{micro}} = R_{\text{micro}} = F1_{\text{micro}} = \text{Accuracy}$$

`recall_weighted` equals accuracy for the same reason. **Five of the ten columns will print
the identical number.** That is correct. The two that actually separate models on
41:1-imbalanced traffic are **`f1_macro`** (every attack class counts equally) and
**`f1_weighted`** (reliability under the real class mix).

In [ ]:
%%writefile /kaggle/working/proj/metrics.py
import os, csv, json
from pathlib import Path
import numpy as np
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

METRIC_KEYS = ["accuracy",
               "precision_macro", "precision_micro", "precision_weighted",
               "recall_macro",    "recall_micro",    "recall_weighted",
               "f1_macro",        "f1_micro",        "f1_weighted"]


def compute_metrics(y_true, y_pred, num_classes):
    """All 10 metrics from one pass. labels=arange pins the class axis so a class the
    model never predicts still occupies its column in the macro average."""
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    labels = np.arange(num_classes)
    out = {"accuracy": float(accuracy_score(y_true, y_pred))}
    for avg in ("macro", "micro", "weighted"):
        p, r, f, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, labels=labels, zero_division=0)
        out[f"precision_{avg}"] = float(p)
        out[f"recall_{avg}"]    = float(r)
        out[f"f1_{avg}"]        = float(f)
    return out


def atomic_write_json(path, obj):
    path = Path(path); tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(obj, indent=2)); os.replace(tmp, path)


def append_csv(path, row, columns):
    path = Path(path); new = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        if new: w.writeheader()
        w.writerow(row)


def save_round_artifacts(round_idx, y_true, y_pred, y_prob, num_classes,
                         class_names, out_dir, extra=None, save_prob=False, metrics=None,
                         is_best=False):
    """y_pred is written every round (10.7 MB as int8). y_prob is 10.7M x 16 floats —
    344 MB per round, so 50 rounds would blow Kaggle's 20 GB output cap. It is written
    only for the best-f1_macro round and the final round (deviation 24).

    `metrics` takes an already-computed dict: the caller needs f1_macro to decide
    `save_prob`, and recomputing it here meant three more sklearn passes over 10.7M rows
    every single round."""
    out_dir = Path(out_dir); extra = extra or {}
    m = dict(metrics) if metrics is not None else compute_metrics(y_true, y_pred, num_classes)
    m["round"] = int(round_idx); m.update(extra)

    atomic_write_json(out_dir / "metrics" / f"round_{round_idx:03d}.json", m)
    append_csv(out_dir / "metrics" / "history.csv", m,
               columns=["round", *METRIC_KEYS, *extra.keys()])

    np.savez_compressed(out_dir / "preds" / f"round_{round_idx:03d}.npz",
                        y_pred=y_pred.astype(np.int8))
    if save_prob:
        # NOT savez_compressed: zlib on 344 MB of fp16 softmax costs a minute or more of
        # wall clock while rank 1 sits in the barrier.
        path = out_dir / "preds" / f"prob_round_{round_idx:03d}.npz"
        np.savez(path, y_prob=y_prob.astype(np.float16))
        # "best and final" means two files, not one per improving round. f1_macro improves
        # on most rounds early, so without this the run keeps every superseded copy —
        # 344 MB each, up to 16 GB by round 50. A new best supersedes the old one; the
        # final round is written after the last possible best, so it survives.
        if is_best:
            for old in Path(out_dir / "preds").glob("prob_round_*.npz"):
                if old != path:
                    old.unlink()
    yt = out_dir / "preds" / "y_true.npy"
    if not yt.exists():
        np.save(yt, y_true.astype(np.int8))          # identical every round — stored once

    np.save(out_dir / "confusion" / f"round_{round_idx:03d}.npy",
            confusion_matrix(y_true, y_pred, labels=np.arange(num_classes)))
    with open(out_dir / "reports" / f"round_{round_idx:03d}.txt", "w") as f:
        f.write(classification_report(y_true, y_pred, labels=np.arange(num_classes),
                                      target_names=class_names, digits=4, zero_division=0))
    return m

In [ ]:
%%writefile /kaggle/working/proj/ckpt.py
"""The resume contract: kill the run at any instant, lose at most one round."""
import os, json, glob, shutil, hashlib, random
from pathlib import Path
import numpy as np, torch

SUBDIRS = ("checkpoints", "metrics", "preds", "confusion", "reports", "logs")


def run_dir(run_name):
    d = Path("/kaggle/working/runs") / run_name
    for s in SUBDIRS: (d / s).mkdir(parents=True, exist_ok=True)
    return d


def resolve_resume(run_name):
    """working/ first, then any attached input (a previous kernel's committed output).
    /kaggle/working is wiped when a NEW session starts, so cross-session resume depends
    on the dead run's output being attached as a data source."""
    d = run_dir(run_name)
    local = d / "checkpoints" / "last.pt"
    if local.exists():
        return local
    # Recursive: Kaggle nests an attached notebook output three levels deep, at
    # /kaggle/input/notebooks/<owner>/<slug>/runs/... A fixed number of "*" segments
    # missed it and the run silently restarted from round 0.
    pats = [f"/kaggle/input/**/runs/{run_name}/checkpoints",
            f"/kaggle/input/**/checkpoints"]
    for pat in pats:
        for src in sorted(glob.glob(pat, recursive=True)):
            if os.path.exists(os.path.join(src, "last.pt")):
                print(f"[resume] importing checkpoints from {src}", flush=True)
                for f in glob.glob(os.path.join(src, "*")):
                    shutil.copy2(f, d / "checkpoints")
                for sub in ("metrics", "preds", "confusion", "reports"):
                    peer = Path(src).parent / sub
                    if peer.is_dir():
                        for f in glob.glob(str(peer / "*")):
                            shutil.copy2(f, d / sub)
                # channel_mask.json and extractor_init.pt are the two files that make a
                # cross-session resume of build 2 valid: without the frozen §4.8 state the
                # resumed network would not be the one the mask was selected against.
                for side in ("channel_mask.json", "extractor_init.pt"):
                    fm = Path(src).parent / side
                    if fm.exists(): shutil.copy2(fm, d / side)
                return d / "checkpoints" / "last.pt"
    return None


def fingerprint(cfg, keys=("num_classes", "n_features", "n_selected", "batch_per_gpu",
                           "world_size", "run_name", "d_model", "patch_len")):
    """Shape-affecting config only. A changed lr may resume; a changed n_selected may not.
    n_features is in here for build 2: it is the extractor's input width, fixed at 66, and
    a checkpoint taken at another width describes a different §4.8 stack entirely."""
    payload = json.dumps({k: cfg.get(k) for k in keys}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def atomic_save(obj, path):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp); os.replace(tmp, path)          # os.replace is atomic on POSIX


def rng_state():
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}


def set_rng_state(s):
    random.setstate(s["python"]); np.random.set_state(s["numpy"])
    torch.set_rng_state(s["torch"].cpu() if hasattr(s["torch"], "cpu") else s["torch"])
    if s.get("cuda") is not None: torch.cuda.set_rng_state_all(s["cuda"])


def save_round(model, opt, scaler, sched, rnd, cfg, metrics, run_name):
    d = run_dir(run_name) / "checkpoints"
    core = getattr(model, "module", model)               # unwrap DDP -> portable state
    blob = {"round": int(rnd), "model": core.state_dict(), "optim": opt.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "sched": sched.state_dict() if sched is not None else None,
            "rng": rng_state(), "cfg": cfg,
            "fingerprint": fingerprint(cfg), "metrics": metrics}
    atomic_save(blob, d / f"ckpt_round_{rnd:03d}.pt")
    atomic_save(blob, d / "last.pt")
    return d / f"ckpt_round_{rnd:03d}.pt"


def load_for_resume(model, opt, scaler, sched, cfg, run_name, device):
    """Returns the round to start from; 0 when there is nothing to resume."""
    d = run_dir(run_name)
    last = d / "checkpoints" / "last.pt"
    if not last.exists():
        print("[resume] no checkpoint found — starting from round 0", flush=True); return 0
    ck = torch.load(last, map_location="cpu", weights_only=False)
    if ck.get("fingerprint") != fingerprint(cfg):
        raise RuntimeError(
            f"Checkpoint fingerprint {ck.get('fingerprint')} != current {fingerprint(cfg)}. "
            "A shape-affecting config changed. Rename CFG.run_name to start a new run, "
            "or restore the original config to resume this one.")
    getattr(model, "module", model).load_state_dict(ck["model"])
    opt.load_state_dict(ck["optim"])
    if scaler is not None and ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
    if sched  is not None and ck.get("sched"):  sched.load_state_dict(ck["sched"])
    try: set_rng_state(ck["rng"])
    except Exception as e: print("[resume] RNG restore skipped:", e, flush=True)
    start = int(ck["round"]) + 1
    print(f"[resume] loaded round {ck['round']} -> continuing at round {start}", flush=True)
    return start

In [ ]:
%%writefile /kaggle/working/proj/train_worker.py
import os, sys, json, time, math
from datetime import timedelta
import numpy as np, torch, torch.nn as nn, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast, GradScaler

sys.path.insert(0, "/kaggle/working")
from proj import ckpt as C
from proj.model import build_model
from proj.metrics import save_round_artifacts

TIMEOUT_MIN = 60


def ddp_setup(rank, world):
    os.environ.setdefault("MASTER_ADDR", "127.0.0.1")
    os.environ["OMP_NUM_THREADS"] = "1"
    torch.set_num_threads(1)
    # set_device BEFORE init: otherwise every rank builds its NCCL communicator on cuda:0.
    torch.cuda.set_device(rank)
    # PyTorch's NCCL default is 10 min; rank 1 sits in the barrier for rank 0's whole
    # 10.7M-row evaluation. Exceeding the timeout aborts the run with no traceback.
    dist.init_process_group("nccl", rank=rank, world_size=world,
                            timeout=timedelta(minutes=int(TIMEOUT_MIN)))


def load_shard(path_X, path_y, rank, world, device):
    """This rank's slice of train, resident on its GPU as fp16. Both ranks take exactly
    n_per rows: a ragged split deadlocks the first collective after the short rank ends."""
    X = np.load(path_X, mmap_mode="r"); Y = np.load(path_y, mmap_mode="r")
    n_per = X.shape[0] // world
    lo, hi = rank * n_per, (rank + 1) * n_per
    chunks_x, chunks_y = [], []
    for i in range(lo, hi, 4_000_000):
        j = min(i + 4_000_000, hi)
        chunks_x.append(torch.from_numpy(np.asarray(X[i:j])).to(device))
        chunks_y.append(torch.from_numpy(np.asarray(Y[i:j]).astype(np.int64)).to(device))
    return torch.cat(chunks_x), torch.cat(chunks_y)


def load_test_resident(path_X, path_y, device):
    """Rank 0 keeps the whole test set on its GPU as fp16, loaded once: 10,761,343 x 66
    fp16 = 1.32 GB. Fancy-indexing a memmap per batch is what used to make eval slow."""
    X = np.load(path_X, mmap_mode="r"); Y = np.load(path_y, mmap_mode="r")
    chunks = []
    for i in range(0, X.shape[0], 4_000_000):
        j = min(i + 4_000_000, X.shape[0])
        chunks.append(torch.from_numpy(np.asarray(X[i:j])).to(device))
    return torch.cat(chunks), np.asarray(Y).astype(np.int8)


@torch.inference_mode()
def evaluate_full(model, Xte, prob_buf, cfg):
    """Rank 0 only. The ENTIRE test set, exactly once, in file order — no
    DistributedSampler, so no padded or dropped tail."""
    model.eval()
    n, B = Xte.shape[0], cfg["eval_batch"]
    preds = torch.empty(n, dtype=torch.int8, device=Xte.device)
    for i in range(0, n, B):
        j = min(i + B, n)
        with autocast("cuda", dtype=torch.float16, enabled=cfg["amp"]):
            logits = model(Xte[i:j].float())
        pr = torch.softmax(logits.float(), dim=1)
        prob_buf[i:j] = pr.to(torch.float16)
        preds[i:j] = pr.argmax(1).to(torch.int8)
    model.train()
    return preds.cpu().numpy()


def main_worker(rank, world, cfg):
    global TIMEOUT_MIN
    TIMEOUT_MIN = cfg.get("nccl_timeout_min", 60)
    ddp_setup(rank, world)
    device = torch.device(f"cuda:{rank}")
    is_main = rank == 0
    torch.manual_seed(cfg["seed"] + rank); np.random.seed(cfg["seed"] + rank)
    torch.backends.cudnn.benchmark = True

    Xs, Ys = load_shard(cfg["train_X"], cfg["train_y"], rank, world, device)
    if is_main:
        print(f"[rank0] shard {tuple(Xs.shape)} fp16 = "
              f"{Xs.element_size()*Xs.nelement()/2**30:.2f} GB resident", flush=True)

    # Build 4 loads NO extractor_init.pt and NO channel_mask.json: both describe a §4.8
    # projection this model does not contain. Nothing is inherited by design.
    model = build_model(cfg, cfg["n_features"]).to(device)
    if is_main:
        print(f"[rank0] params: {sum(p.numel() for p in model.parameters()):,}   "
              f"DAGSNet only — input (B, {cfg['patch_len']}, {model.k}), no extractor, no CMSO",
              flush=True)

    model = DDP(model, device_ids=[rank], output_device=rank,
                gradient_as_bucket_view=True, find_unused_parameters=False)

    # DDP FIRST, then compile — the order DDPOptimizer needs to break the graph on
    # gradient-bucket boundaries. Default mode, not "reduce-overhead": CUDA graphs
    # interact badly with DDP + GradScaler. The trial step proves the whole fwd+bwd path
    # really compiles; anything wrong drops back to eager and the run still happens. Both
    # ranks run the identical trial, so DDP stays in step either way.
    if cfg.get("compile_model", False):
        eager = model
        try:
            model = torch.compile(model)
            xt = torch.zeros(cfg["batch_per_gpu"], cfg["n_features"], device=device)
            yt = torch.zeros(cfg["batch_per_gpu"], dtype=torch.long, device=device)
            with autocast("cuda", dtype=torch.float16, enabled=cfg["amp"]):
                lt = model(xt)
            torch.nn.functional.cross_entropy(lt.float(), yt).backward()
            model.zero_grad(set_to_none=True)
            if is_main:
                print("[rank0] torch.compile active (trial fwd+bwd passed)", flush=True)
        except Exception as e:
            model = eager
            model.zero_grad(set_to_none=True)
            if is_main:
                print(f"[rank0] torch.compile failed ({type(e).__name__}: {e}) — "
                      f"running eager", flush=True)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scaler = GradScaler("cuda", enabled=cfg["amp"])

    def lr_at(rnd):
        w = cfg["warmup_rounds"]
        if rnd < w: return (rnd + 1) / max(w, 1)
        prog = (rnd - w) / max(cfg["rounds"] - w, 1)
        return 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    crit = nn.CrossEntropyLoss()

    if cfg["resume"] and is_main:
        C.resolve_resume(cfg["run_name"])
    dist.barrier()
    start_round = C.load_for_resume(model, opt, scaler, sched, cfg,
                                    cfg["run_name"], device) if cfg["resume"] else 0

    B = cfg["batch_per_gpu"]
    steps = Xs.shape[0] // B

    # ---- observability: rank 0 owns the single W&B run ------------------------
    # Kaggle publishes a kernel's output only when it stops, so without this the run is
    # invisible for its whole 4 h. The JSONL heartbeat is written too and is committed
    # with the output, so evidence survives an account or network problem.
    HEART = C.run_dir(cfg["run_name"]) / "logs" / "heartbeat.jsonl"
    WB = None
    if is_main:
        try:
            import wandb
            WB = wandb.init(entity=cfg["wandb_entity"], project=cfg["wandb_project"],
                            id=cfg["wandb_run_id"], name=cfg["run_name"], config=cfg,
                            resume="allow", mode="online",
                            dir=str(C.run_dir(cfg["run_name"])))
            WB.define_metric("progress/global_step")
            WB.define_metric("*", step_metric="progress/global_step")
            print(f"[rank0] W&B run: {WB.url}", flush=True)
        except Exception as e:
            print(f"[rank0] W&B unavailable ({type(e).__name__}: {e}) — "
                  f"the JSONL heartbeat still records everything", flush=True)

    def emit(rec, payload=None, gstep=None):
        with HEART.open("a") as fh:
            fh.write(json.dumps(rec) + "\n")
        print("  " + json.dumps(rec), flush=True)      # flush: a killed kernel loses buffers
        if WB is not None and payload is not None:
            WB.log(payload, step=gstep)

    if is_main:
        print(f"[rank0] {steps:,} steps/epoch/rank  (global batch "
              f"{B * world * cfg['grad_accum']})", flush=True)

    Xte = y_true = prob_buf = None
    if is_main:
        Xte, y_true = load_test_resident(cfg["test_X"], cfg["test_y"], device)
        prob_buf = torch.empty((Xte.shape[0], cfg["num_classes"]),
                               dtype=torch.float16, device=device)
        print(f"[rank0] test {tuple(Xte.shape)} fp16 = "
              f"{Xte.element_size()*Xte.nelement()/2**30:.2f} GB resident", flush=True)

    deadline = cfg.get("deadline_ts", float("inf"))
    best_f1 = -1.0
    hist_p = C.run_dir(cfg["run_name"]) / "metrics" / "history.csv"
    if hist_p.exists():
        import csv as _csv
        with open(hist_p) as f:
            best_f1 = max([float(r["f1_macro"]) for r in _csv.DictReader(f)] or [-1.0])

    LOG_EVERY = cfg.get("heartbeat_every", 250)
    outcome = "running"

    for rnd in range(start_round, cfg["rounds"]):
        t0 = time.time(); model.train()
        running = torch.zeros(2, device=device)
        skipped = win_skip = win_ok = 0
        aborted = False
        step_offset = rnd * steps

        for ep in range(cfg["epochs_per_round"]):
            g = torch.Generator(device=device)
            g.manual_seed(cfg["seed"] * 100003 + rnd * 1009 + ep)   # same stream on both ranks
            perm = torch.randperm(Xs.shape[0], generator=g, device=device)
            opt.zero_grad(set_to_none=True)
            for s in range(steps):
                idx = perm[s * B:(s + 1) * B]
                xb = Xs[idx].float(); yb = Ys[idx]
                with autocast("cuda", dtype=torch.float16, enabled=cfg["amp"]):
                    logits = model(xb)
                # NaN MECHANISM 1. Cross-entropy in fp32, OUTSIDE autocast. A softmax over
                # 16 half-precision logits can round a class probability to exactly 0, and
                # log(0) is -inf. The tensor is (B, 16), so this costs nothing.
                loss = crit(logits.float(), yb) / cfg["grad_accum"]
                scaler.scale(loss).backward()
                if (s + 1) % cfg["grad_accum"] == 0:
                    scaler.unscale_(opt)
                    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip"])
                    # NaN MECHANISM 3. GradScaler discards a non-finite step and reports
                    # NOTHING: a run can throw away most of its batches while printing a
                    # falling loss. The scale halving is the only observable signal, so
                    # read it and count. One host read per step on a scalar already on the
                    # host — CUDA, unlike XLA, does not drain a queue for this.
                    prev = scaler.get_scale()
                    scaler.step(opt); scaler.update()
                    if scaler.get_scale() < prev:
                        skipped += 1; win_skip += 1
                    else:
                        win_ok += 1
                    opt.zero_grad(set_to_none=True)
                running += torch.stack([loss.detach() * cfg["grad_accum"],
                                        torch.ones((), device=device)])

                want_abort = False
                if is_main and (s + 1) % LOG_EVERY == 0:
                    gstep = step_offset + s + 1
                    tot = max(win_ok + win_skip, 1)
                    skip_pct = 100.0 * win_skip / tot
                    gn = float(gnorm)
                    with torch.no_grad():
                        mw = float(max(p.abs().max() for p in model.parameters()))
                    rec = {"round": rnd, "step": s + 1, "of": steps, "global_step": gstep,
                           "loss": round(float(loss) * cfg["grad_accum"], 5),
                           "grad_norm": round(gn, 5) if math.isfinite(gn) else "nonfinite",
                           "skipped_in_window": win_skip, "skip_pct": round(skip_pct, 2),
                           "max_abs_weight": round(mw, 4),
                           "scaler_scale": scaler.get_scale(),
                           "lr": opt.param_groups[0]["lr"],
                           "elapsed_s": round(time.time() - t0, 1)}
                    emit(rec, {"progress/round": rnd, "progress/round_step": s + 1,
                               "progress/global_step": gstep,
                               "train/loss": rec["loss"],
                               "train/grad_norm": gn if math.isfinite(gn) else float("nan"),
                               "train/skip_pct": skip_pct,
                               "train/max_abs_weight": mw,
                               "train/scaler_scale": scaler.get_scale(),
                               "train/lr": opt.param_groups[0]["lr"],
                               "train/global_batch": B * world * cfg["grad_accum"]}, gstep)
                    # In-round abort. A skip-dominated run stays finite while learning
                    # nothing, so the loss tripwire never sees it; a round-level check
                    # would still cost a whole round. Grace period first, so a noisy
                    # start cannot kill a healthy run.
                    want_abort = (s + 1 >= cfg["abort_after"]
                                  and skip_pct > cfg["abort_skip_pct"])
                    if want_abort:
                        emit({"event": "abort", "round": rnd, "step": s + 1,
                              "skip_pct": round(skip_pct, 2),
                              "reason": "GradScaler discarded most of the window"})
                    win_ok = win_skip = 0

                # THE DECISION IS COLLECTIVE, AND EVERY RANK REACHES IT.
                # Only rank 0 computes the skip rate, but if it broke out of the loop
                # alone, rank 1's next backward() would block forever inside DDP's
                # gradient all-reduce. So both ranks meet here on the same step and take
                # the same branch. Same reason the budget check below uses all_reduce
                # instead of each rank reading its own clock.
                if (s + 1) % LOG_EVERY == 0:
                    flag = torch.tensor([1.0 if (is_main and want_abort) else 0.0],
                                        device=device)
                    dist.all_reduce(flag, op=dist.ReduceOp.MAX)
                    if flag.item() > 0:
                        aborted = True
                        break
            if aborted:
                break

        sched.step()
        dist.all_reduce(running, op=dist.ReduceOp.SUM)
        train_loss = (running[0] / running[1]).item()

        if aborted:
            if is_main:
                (C.run_dir(cfg["run_name"]) / "logs" / "diverged.json").write_text(
                    json.dumps({"round": int(rnd), "reason": "skip-rate abort",
                                "skipped_steps": int(skipped),
                                "train_loss": str(train_loss)}, indent=2))
                print(f"[abort] round {rnd:03d}: majority of steps discarded. Stopping.",
                      flush=True)
            outcome = "aborted"
            break

        # NaN MECHANISM 2. train_loss comes out of an all_reduce(SUM), so both ranks hold
        # the SAME value and break together without another collective. Build 2 run 1 went
        # non-finite at round 25 and kept going for 20 more rounds: 4.1 h of GPU, 20 NaN
        # checkpoints, and last.pt overwritten with NaN weights so it could not be resumed.
        # Breaking BEFORE eval and BEFORE save_round leaves last.pt on the last valid round.
        if not math.isfinite(train_loss):
            if is_main:
                print(f"[diverged] round {rnd:03d}: train_loss = {train_loss}. Stopping now; "
                      f"last.pt still holds round {rnd - 1}, the last valid one.", flush=True)
                (C.run_dir(cfg["run_name"]) / "logs" / "diverged.json").write_text(
                    json.dumps({"round": int(rnd), "train_loss": str(train_loss),
                                "last_valid_round": int(rnd) - 1,
                                "skipped_steps": int(skipped),
                                "reason": "non-finite training loss"}, indent=2))
            outcome = "diverged"
            break

        if is_main:
            y_pred = evaluate_full(model.module, Xte, prob_buf, cfg)
            from proj.metrics import compute_metrics
            m0 = compute_metrics(y_true, y_pred, cfg["num_classes"])
            f1m = m0["f1_macro"]
            is_best = f1m > best_f1; best_f1 = max(best_f1, f1m)
            save_prob = is_best or rnd == cfg["rounds"] - 1
            y_prob = prob_buf.cpu().numpy() if save_prob else None
            round_s = time.time() - t0
            m = save_round_artifacts(
                rnd, y_true, y_pred, y_prob, cfg["num_classes"], cfg["class_names"],
                C.run_dir(cfg["run_name"]), metrics=m0, is_best=is_best,
                extra={"train_loss": train_loss, "lr": opt.param_groups[0]["lr"],
                       "seconds": round(round_s, 1),
                       "skipped_steps": int(skipped),
                       "samples_per_sec": round(steps * B * world * cfg["epochs_per_round"]
                                                / max(round_s, 1e-9)),
                       "peak_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2)},
                save_prob=save_prob)
            C.save_round(model, opt, scaler, sched, rnd, cfg, m, cfg["run_name"])
            print(f"[round {rnd:03d}] loss {train_loss:.4f}  acc {m['accuracy']:.4f}  "
                  f"F1_mac {m['f1_macro']:.4f}  F1_wtd {m['f1_weighted']:.4f}  "
                  f"{m['seconds']}s  {m['samples_per_sec']:,}/s  skipped {skipped}  "
                  f"peak {m['peak_gb']}GB" + ("  <- best" if is_best else ""), flush=True)
            if WB is not None:
                WB.log({"progress/completed_round": rnd,
                        "progress/global_step": (rnd + 1) * steps,
                        "round/duration_s": round_s,
                        "round/skipped_steps": int(skipped),
                        "round/train_loss": train_loss,
                        **{f"eval/{k}": v for k, v in m0.items()}},
                       step=(rnd + 1) * steps)
            del y_pred, y_prob
            torch.cuda.reset_peak_memory_stats()

        dist.barrier()      # after the atomic write: a kill during eval leaves a valid last.pt

        # Session budget: only begin a round we can finish. The decision goes through
        # all_reduce — a rank reading its own clock would break alone and deadlock the next
        # collective.
        probe = torch.tensor([time.time() - t0], device=device)
        dist.all_reduce(probe, op=dist.ReduceOp.MAX)
        round_s = probe.item()
        over = torch.tensor([1.0 if time.time() + 1.15 * round_s > deadline else 0.0],
                            device=device)
        dist.all_reduce(over, op=dist.ReduceOp.MAX)
        if over.item() > 0 and rnd + 1 < cfg["rounds"]:
            if is_main:
                left = (deadline - time.time()) / 3600
                print(f"[budget] stopping cleanly after round {rnd:03d}: next round needs "
                      f"~{round_s/60:.1f} min, {left*60:.1f} min left in the budget. "
                      f"Re-push with this run's output attached to resume at round {rnd+1}.",
                      flush=True)
                (C.run_dir(cfg["run_name"]) / "logs" / "stopped_early.json").write_text(
                    json.dumps({"last_round": int(rnd), "next_round": int(rnd) + 1,
                                "reason": "session wall-clock budget",
                                "round_seconds": round(round_s, 1)}, indent=2))
            outcome = "budget"
            break

    if is_main and WB is not None:
        WB.summary["outcome"] = outcome
        WB.finish()
    dist.destroy_process_group()

In [ ]:
# ── Cell 15 · launch ─────────────────────────────────────────────────────────
import socket, importlib, torch.multiprocessing as mp

os.environ["NCCL_P2P_DISABLE"] = "1"        # PCIe P2P is blocked in the Kaggle container:
os.environ["NCCL_IB_DISABLE"]  = "1"        # without this init_process_group hangs until
os.environ["NCCL_ASYNC_ERROR_HANDLING"] = "1"        # the timeout instead of failing fast
os.environ["TORCH_NCCL_ASYNC_ERROR_HANDLING"] = "1"  # the name torch >= 2.2 actually reads
with socket.socket() as s:
    s.bind(("", 0)); os.environ["MASTER_PORT"] = str(s.getsockname()[1])

DEADLINE = NB_T0 + CFG.max_hours * 3600
print(f"prep took {(time.time()-NB_T0)/60:.1f} min; "
      f"{(DEADLINE-time.time())/3600:.2f} h of session budget left for training")

cfg = asdict(CFG)
cfg.update(deadline_ts=DEADLINE, class_names=CLASS_NAMES, num_classes=NUM_CLASSES,
           train_X=str(XTR), train_y=str(YTR), test_X=str(XTE), test_y=str(YTE))
(RUN_DIR / "config.json").write_text(json.dumps(cfg, indent=2))

# Build 4's invariants. There is no CMSO mask and no fused axis to check; what must hold
# is that every input column reaches DAGSNet and that the patch reshape is exact.
assert cfg["n_features"] == len(FEATURES) == 66, \
    "DAGSNet must see every input column — build 4 selects nothing away"
assert cfg["n_features"] % CFG.patch_len == 0, \
    f"{cfg['n_features']} features do not divide into patches of {CFG.patch_len}"
assert not (RUN_DIR / "channel_mask.json").exists(), \
    "a CMSO mask has no meaning in build 4 — it describes a §4.8 projection that is absent"

steps = (N_TRAIN // CFG.world_size) // CFG.batch_per_gpu
print(f"{steps:,} steps/epoch/rank x {CFG.world_size} ranks x {CFG.rounds} rounds")
print(f"inputs: {len(FEATURES)} -> DAGSNet input (B, {CFG.patch_len}, "
      f"{cfg['n_features']//CFG.patch_len})   classes: {NUM_CLASSES}")
print(f"free on /kaggle/working: {shutil.disk_usage('/kaggle/working').free/2**30:.1f} GB "
      f"(50 ckpts + 50 y_pred must fit)")

import proj.train_worker as W
importlib.reload(W)

t0 = time.time()
try:
    mp.spawn(W.main_worker, args=(CFG.world_size, cfg), nprocs=CFG.world_size, join=True)
    hist = RUN_DIR / "metrics" / "history.csv"
    done = (sum(1 for _ in open(hist)) - 1) if hist.exists() else 0
    for name, msg in (("stopped_early.json", "stopped on the session budget"),
                      ("diverged.json", "stopped by a tripwire")):
        p = RUN_DIR / "logs" / name
        if p.exists():
            print(f"\n{msg} after {done}/{CFG.rounds} rounds "
                  f"in {(time.time()-t0)/3600:.2f} h:")
            print(json.dumps(json.loads(p.read_text()), indent=2))
            print("Re-push with this kernel's output attached as a data source to resume.")
            break
    else:
        print(f"\nall {CFG.rounds} rounds complete in {(time.time()-t0)/3600:.2f} h")
except Exception:
    import traceback; traceback.print_exc()
    print("\nRun failed. A ProcessRaisedException wraps the CHILD's traceback — read the "
          "inner one. Fix the cause, re-run the %%writefile cell, then re-run this cell: "
          "training resumes from the last completed round, nothing is retrained.")
    raise

In [ ]:
# ── Cell 16 · results ────────────────────────────────────────────────────────
# A run stopped inside round 0 writes no history.csv. Reading it unguarded raises
# FileNotFoundError, papermill turns that into an execution error, and Kaggle marks the
# whole kernel FAILED even though the tripwire did exactly its job — that mislabelled two
# runs in build 3. Print the evidence instead of raising.
HIST = RUN_DIR / "metrics" / "history.csv"
hist = final = None
if not HIST.exists():
    print("no completed round — nothing to plot.")
    for name in ("diverged.json", "stopped_early.json"):
        p = RUN_DIR / "logs" / name
        if p.exists():
            print(f"\nlogs/{name}:", json.dumps(json.loads(p.read_text()), indent=2))
    for f in sorted((RUN_DIR / "logs").glob("*.jsonl")):
        print(f"\n--- {f.name} (last 20) ---")
        print("".join(f.read_text().splitlines(keepends=True)[-20:]))
else:
    import pandas as pd
    from proj.metrics import METRIC_KEYS

    hist = (pd.read_csv(HIST).drop_duplicates(subset="round", keep="last")
              .sort_values("round").reset_index(drop=True))
    print(f"{len(hist)} / {CFG.rounds} rounds completed")
    display(hist[["round", *METRIC_KEYS]].round(5))

    ax = hist.plot(x="round", y=["accuracy", "f1_macro", "f1_weighted"],
                   marker="o", figsize=(10, 4.5))
    ax.set_title("Build 4 (DAGSNet only) — full 10,761,343-row test set")
    ax.set_xlabel("round"); ax.grid(alpha=.3)

    best, final = hist.loc[hist["f1_macro"].idxmax()], hist.iloc[-1]
    print(f"\nbest round : {int(best['round'])}  f1_macro {best['f1_macro']:.5f}")
    print(f"final round: {int(final['round'])}  f1_macro {final['f1_macro']:.5f}")
    print(f"\nwall time  : {hist['seconds'].sum()/3600:.2f} h over {len(hist)} rounds "
          f"({hist['seconds'].mean()/60:.1f} min/round, "
          f"{hist['samples_per_sec'].mean():,.0f} samples/s)")
    if "skipped_steps" in hist:
        print(f"steps discarded by GradScaler: {int(hist['skipped_steps'].sum()):,} total")

    print(f"\n{'='*72}\nFINAL 10 METRICS (round {int(final['round'])})\n{'='*72}")
    for k in METRIC_KEYS:
        print(f"  {k:<22} {final[k]:.6f}")
    print("\nNote: precision_micro = recall_micro = f1_micro = recall_weighted = accuracy.")
    print("This identity holds for any single-label multi-class problem — it is not a bug.")

    print(f"\n{'='*72}\nPER-CLASS REPORT (round {int(final['round'])})\n{'='*72}")
    print(open(RUN_DIR / "reports" / f"round_{int(final['round']):03d}.txt").read())

In [ ]:
# ── Cell 17 · confusion matrix ───────────────────────────────────────────────
if final is None:
    print("no completed round — no confusion matrix to draw.")
else:
    import matplotlib.pyplot as plt

    cm = np.load(RUN_DIR / "confusion" / f"round_{int(final['round']):03d}.npy")
    cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cmn, cmap="magma", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"Row-normalised confusion — round {int(final['round'])}, full test set")
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            if cmn[i, j] > 0.005:
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cmn[i, j] < 0.6 else "black")
    fig.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

    for name in ("trafficCongestionSybil", "timeDelayAttack"):
        if name in CLASS_NAMES:
            i = CLASS_NAMES.index(name)
            rec = cm[i, i] / max(cm[i].sum(), 1)
            prec = cm[i, i] / max(cm[:, i].sum(), 1)
            f1 = 2 * prec * rec / max(prec + rec, 1e-12)
            print(f"{name:<26} precision {prec:.4f}  recall {rec:.4f}  f1 {f1:.4f}")

## How to read these numbers

**This is an ablation, and it only answers one question**: what does the §4.8 extractor plus
§4.9 CMSO add on top of DAGSNet? Compare `f1_macro` here against **0.83081** (build 2 run 1,
round 0) and **0.82422** (build 2 run 3, round 0) — both the full pipeline at the same batch,
same optimiser, same schedule, on the same split.

Three readings, and each means something different:

* **Build 4 clearly lower** — the extractor earns its 332,928 extra parameters.
* **Build 4 comparable** — the extractor is not contributing on this data, and the score comes
  from DAGSNet plus the features themselves. Section 3.4 of the aggregate report already found
  the ViT saturated (`max prob = 1.0000` from round 0), which would make this the expected
  outcome.
* **Build 4 higher** — the extractor is actively hurting, most likely through the instability
  documented in builds 2 and 3.

**Every caveat in CONTEXT.md §6 still applies here** and must accompany any number from this
run: the time-based split, the attack-free benign stream, the 3,338,358 dropped ambiguous rows,
the `session`-group leakage that inflates `trafficCongestionSybil`, `timeDelayAttack` as the
hardest class, the 41:1 imbalance that makes `accuracy` nearly meaningless next to `f1_macro`,
and the 88% of test flows that are a single message.

**Do not put these numbers in the same table as the paper's.** Khan et al. report binary
classification on CIC-IDS 2017 / CAN with metrics (specificity, NPV, MCC, FPR, FNR) that need a
TN definition 16-class classification does not have. What is inherited from the paper is the
method, not the numbers.

One caveat specific to build 4: reshaping 66 features into 11 positions × 6 channels imposes an
ordering the columns do not really have. The full pipeline does the same thing at the same place
(`PatchEmbed` chunks the DWT output identically), so the comparison is fair — but neither build
should be read as evidence that convolution over the feature axis is *appropriate* for tabular
data. That is a separate question this run does not test.